In [44]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [45]:
df = pd.read_csv("output/cleaned_movie_released_nonzero.csv")
released = df.copy()
released.head()


,Unnamed: 0,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,budget,...,production_countries_United States of America,production_countries_France,production_countries_Japan,production_countries_Germany,production_countries_United Kingdom,spoken_languages_English,spoken_languages_French,spoken_languages_Spanish,spoken_languages_Japanese,spoken_languages_German
0,0,Inception,8.364,34495,Released,2010-07-15,825532764,148,0,160000000,...,1,0,0,0,1,1,1,0,1,0
1,1,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,0,165000000,...,1,0,0,0,1,1,0,0,0,0
2,2,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,0,185000000,...,1,0,0,0,1,1,0,0,0,0
3,3,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,0,237000000,...,1,0,0,0,1,1,0,1,0,0
4,4,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,0,220000000,...,1,0,0,0,0,1,0,0,0,0


Input Features

In [46]:
input_columns = [
    # 'release_date', 
    # 'release_season', 
    'release_year', 
    'release_quarter', 
    'release_month',
    'budget'
]

for prefix in ['genre_', 'production_companies_', 'production_countries_', 'spoken_languages_']:
    input_columns.extend([col for col in df.columns if col.startswith(prefix)])

X = df[input_columns]
X = X.copy()
X.columns = X.columns.str.replace(' ', '_')


In [47]:
# Add these features to enhance predictive power

# Budget-based features
X['log_budget'] = np.log1p(df['budget'])  # Log transform to handle skewness
X['budget_per_runtime'] = df['budget'] / df['runtime'].replace(0, 1)  # Cost per minute

# # Competition features
# release_counts = df.groupby(['release_year', 'release_month']).size()
# df['release_competition'] = df.apply(lambda x: release_counts.get((x['release_year'], x['release_month']), 0), axis=1)
# X['release_competition'] = df['release_competition']

# Genre combinations (for multi-genre films)
genre_cols = [col for col in df.columns if col.startswith('genre_')]
X['genre_count'] = df[genre_cols].sum(axis=1)

In [48]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10458 entries, 0 to 10457
Data columns (total 41 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   release_year                                   10458 non-null  int64  
 1   release_quarter                                10458 non-null  int64  
 2   release_month                                  10458 non-null  int64  
 3   budget                                         10458 non-null  int64  
 4   genre_Action                                   10458 non-null  int64  
 5   genre_Adventure                                10458 non-null  int64  
 6   genre_Animation                                10458 non-null  int64  
 7   genre_Comedy                                   10458 non-null  int64  
 8   genre_Crime                                    10458 non-null  int64  
 9   genre_Documentary                              104

Target Outputs

In [49]:
y_revenue = df['revenue']
y_popularity = df['popularity']
y_vote_count = df['vote_count']
y_vote_average = df['vote_average']

Train-Test Split

In [50]:
X_train_revenue, X_test_revenue, y_train_revenue, y_test_revenue = train_test_split(X, y_revenue, test_size=0.1)
X_train_popularity, X_test_popularity, y_train_popularity, y_test_popularity = train_test_split(X, y_popularity, test_size=0.1)
X_train_vote_count, X_test_vote_count, y_train_vote_count, y_test_vote_count = train_test_split(X, y_vote_count, test_size=0.1)
X_train_vote_average, X_test_vote_average, y_train_vote_average, y_test_vote_average = train_test_split(X, y_vote_average, test_size=0.1)

Training the Models

In [54]:
X.to_numpy()

array([[2.01000000e+03, 3.00000000e+00, 7.00000000e+00, ...,
        1.88906844e+01, 1.08108108e+06, 3.00000000e+00],
       [2.01400000e+03, 4.00000000e+00, 1.10000000e+01, ...,
        1.89214560e+01, 9.76331361e+05, 3.00000000e+00],
       [2.00800000e+03, 3.00000000e+00, 7.00000000e+00, ...,
        1.90358664e+01, 1.21710526e+06, 4.00000000e+00],
       ...,
       [2.00900000e+03, 4.00000000e+00, 1.10000000e+01, ...,
        1.45086582e+01, 2.27272727e+04, 2.00000000e+00],
       [2.00700000e+03, 3.00000000e+00, 9.00000000e+00, ...,
        1.34907882e+01, 8.12052809e+03, 2.00000000e+00],
       [2.01200000e+03, 1.00000000e+00, 2.00000000e+00, ...,
        6.21660610e+00, 3.33333333e+01, 1.00000000e+00]])

In [52]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import numpy as np

# Create a baseline model for comparison
baseline_model = LinearRegression()
baseline_model.fit(X_train_revenue.to_numpy(), y_train_revenue.to_numpy())

baseline_pred = baseline_model.predict(X_test_revenue)
baseline_r2 = r2_score(y_test_revenue, baseline_pred)
print(f"Baseline Linear Regression R²: {baseline_r2:.4f}")

# Add more advanced models
models = {
    'GBR': GradientBoostingRegressor(),
    'Random Forest': RandomForestRegressor(n_estimators=100, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100),
    'LightGBM': LGBMRegressor(n_estimators=100)
}

# Train and evaluate each model
results = {}
for name, model in models.items():
    model.fit(X_train_revenue, y_train_revenue)
    y_pred = model.predict(X_test_revenue)
    results[name] = {
        'MAE': mean_absolute_error(y_test_revenue, y_pred),
        'R²': r2_score(y_test_revenue, y_pred)
    }

# Create an ensemble model
estimators = [(name, model) for name, model in models.items()]
ensemble = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge()
)
ensemble.fit(X_train_revenue, y_train_revenue)
ensemble_pred = ensemble.predict(X_test_revenue)
ensemble_r2 = r2_score(y_test_revenue, ensemble_pred)
print(f"Ensemble Model R²: {ensemble_r2:.4f}")

C:\.School\csci_113i\dim_modelling\.venv\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


Baseline Linear Regression R²: 0.5903
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000405 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 961
[LightGBM] [Info] Number of data points in the train set: 9412, number of used features: 39
[LightGBM] [Info] Start training from score 63921740.856247
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000378 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 961
[LightGBM] [Info] Number of data points in the train set: 9412, number of used features: 39
[LightGBM] [Info] Start training from score 63921740.856247
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000937 seconds.
You can set `force_col_wise=true` to

In [13]:
gbr_revenue = GradientBoostingRegressor()
gbr_revenue.fit(X_train_revenue, y_train_revenue)
y_pred_revenue = gbr_revenue.predict(X_test_revenue)

gbr_popularity = GradientBoostingRegressor()
gbr_popularity.fit(X_train_popularity, y_train_popularity)
y_pred_popularity = gbr_popularity.predict(X_test_popularity)

gbr_vote_count = GradientBoostingRegressor()
gbr_vote_count.fit(X_train_vote_count, y_train_vote_count)
y_pred_vote_count = gbr_vote_count.predict(X_test_vote_count)

gbr_vote_average = GradientBoostingRegressor()
gbr_vote_average.fit(X_train_vote_average, y_train_vote_average)
y_pred_vote_average = gbr_vote_average.predict(X_test_vote_average)

ValueError: Input X contains NaN.
GradientBoostingRegressor does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

Evaluation of Models

In [ ]:
print("\nrevenue")
print("mean_absolute_error: ", mean_absolute_error(y_test_revenue, y_pred_revenue))
print("mean_squared_error : ", mean_squared_error(y_test_revenue, y_pred_revenue))
print("r2_score           : ", r2_score(y_test_revenue, y_pred_revenue))

print("\npopularity")
print("mean_absolute_error: ", mean_absolute_error(y_test_popularity, y_pred_popularity))
print("mean_squared_error : ", mean_squared_error(y_test_popularity, y_pred_popularity))
print("r2_score           : ", r2_score(y_test_popularity, y_pred_popularity))

print("\nvote_count")
print("mean_absolute_error: ", mean_absolute_error(y_test_vote_count, y_pred_vote_count))
print("mean_squared_error : ", mean_squared_error(y_test_vote_count, y_pred_vote_count))
print("r2_score           : ", r2_score(y_test_vote_count, y_pred_vote_count))

print("\nvote_average")
print("mean_absolute_error: ", mean_absolute_error(y_test_vote_average, y_pred_vote_average))
print("mean_squared_error : ", mean_squared_error(y_test_vote_average, y_pred_vote_average))
print("r2_score           : ", r2_score(y_test_vote_average, y_pred_vote_average))

In [ ]:
importances_revenue = pd.Series(gbr_revenue.feature_importances_, index=X.columns)
importances_revenue = importances_revenue.sort_values(ascending=False)
print("Revenue Importances:\n", importances_revenue)

importances_popularity = pd.Series(gbr_popularity.feature_importances_, index=X.columns)
importances_popularity = importances_popularity.sort_values(ascending=False)
print("\nPopularity Importances:\n", importances_popularity)

importances_vote_count = pd.Series(gbr_vote_count.feature_importances_, index=X.columns)
importances_vote_count = importances_vote_count.sort_values(ascending=False)
print("\nVote Count Importances:\n", importances_vote_count)

importances_vote_average = pd.Series(gbr_vote_average.feature_importances_, index=X.columns)
importances_vote_average = importances_vote_average.sort_values(ascending=False)
print("\nVote Average Importances:\n", importances_vote_average)

GridSearchCV (Testing Out Different Parameters for GradientBoostingRegressor)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100],
    'learning_rate': [0.1],
    'max_depth': [3, 4, 5],
    'min_samples_leaf': [1, 2, 3, 4],
    'min_samples_split': [2, 3, 4]
}

gbr_cv = GridSearchCV(gbr_revenue, param_grid, cv=5, n_jobs=-1, scoring='neg_mean_squared_error')
gbr_cv.fit(X_train_revenue, y_train_revenue)

In [ ]:
y_pred_revenue = gbr_cv.predict(X_test_revenue)
print(mean_absolute_error(y_test_revenue, y_pred_revenue))
print(mean_squared_error(y_test_revenue, y_pred_revenue))
print(r2_score(y_test_revenue, y_pred_revenue))
print(gbr_cv.best_params_)

In [ ]:
'''
0.5668896280610903
{'criterion': 'friedman_mse',
 'learning_rate': 0.01,
 'max_depth': 5,
 'min_samples_leaf': 1,
 'min_samples_split': 4,
 'n_estimators': 300}

49725670.23436365
1.3831595103615548e+16
0.5661379683786802
{'learning_rate': 0.01,
 'max_depth': 5,
 'min_samples_leaf': 4,
 'min_samples_split': 4,
 'n_estimators': 300}

49470636.75543827
1.386009389191735e+16
0.5652440337240913
{'learning_rate': 0.01,
 'max_depth': 5,
 'min_samples_leaf': 6,
 'min_samples_split': 3,
 'n_estimators': 300}

49858141.79351606
1.4097237970131422e+16
0.5578054258997466
{'learning_rate': 0.01,
 'max_depth': 5,
 'min_samples_leaf': 2,
 'min_samples_split': 4,
 'n_estimators': 300}

49830392.0367167
1.407046087210932e+16
0.558645355500184
{'learning_rate': 0.01,
 'max_depth': 5,
 'min_samples_leaf': 2,
 'min_samples_split': 4,
 'n_estimators': 300}

49847806.99900844
1.4092120649101166e+16
0.5579659432718944
{'learning_rate': 0.01,
 'max_depth': 5,
 'min_samples_leaf': 2,
 'min_samples_split': 4,
 'n_estimators': 300}

49655821.13375789
1.3569761623000274e+16
0.5743510200907369
{'learning_rate': 0.1,
 'max_depth': 3,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 100}

49663860.925762735
1.3863652549843808e+16
0.5651324076573758
{'learning_rate': 0.1,
 'max_depth': 4,
 'min_samples_leaf': 1,
 'min_samples_split': 4,
 'n_estimators': 100}
'''

___
# Claude

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values